# OpenJev: text and image decisions

Use **Qwen3.5-0.8B** to route a support message, score urgency, and classify an image. OpenJev reads the probability of `yes` for each candidate answer and shares prefix computation. It generates **zero answer tokens**.

1. Choose **Runtime → Change runtime type → GPU** (a T4 is enough for this small example, subject to availability).
2. Choose **Runtime → Run all**. The first run installs the package and downloads the model.
3. Edit the message or model settings and rerun the relevant cells. Use **File → Save a copy in Drive** to keep your changes.

No model API key or Drive mount is needed. Without a GPU the default example falls back to CPU and runs more slowly.

[GitHub / README](https://github.com/mjdileep/OpenJev) · [How scoring works](https://github.com/mjdileep/OpenJev/blob/main/docs/architecture.md)

## 1. Install

This installs a pinned OpenJev commit and the Transformers backend. It keeps Colab's installed PyTorch when it meets the package requirements. Start with a fresh runtime; if Colab asks for a restart after installation, restart the session and continue below.

In [ ]:
OPENJEV_REF = "43dc267a80dce7992424ec97d00bfe9a1898f59e"
OPENJEV_PACKAGE = (
    "openjev[cuda,cuda-4bit] @ git+https://github.com/mjdileep/OpenJev.git@" + OPENJEV_REF
)
%pip install -q "{OPENJEV_PACKAGE}" "transformers==5.17.0"

## 2. Load the model once

The default 0.8B model is small enough to use without extra quantization. Enable `USE_4BIT` to try bitsandbytes NF4 on a GPU. Scores can change with quantization.

To try another Hugging Face repository, edit `MODEL_ID` and optionally `MODEL_REVISION`. Keep `ENABLE_IMAGES` enabled for Qwen3.5 models; disable it for other supported text instruction models. After changing these settings, rerun this cell and the examples below.

In [ ]:
# @title Model settings
import gc
import json
from importlib.metadata import version
from pathlib import Path

import torch
from IPython.display import display
from PIL import Image

from openjev import Choice, DecisionEngine, Noul, Score
from openjev.cli import benchmark

MODEL_ID = "Qwen/Qwen3.5-0.8B"  # @param {type:"string"}
MODEL_REVISION = ""  # @param {type:"string"}
USE_4BIT = False  # @param {type:"boolean"}
ENABLE_IMAGES = True  # @param {type:"boolean"}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if USE_4BIT and DEVICE != "cuda":
    raise RuntimeError("Choose a GPU runtime for USE_4BIT, or turn USE_4BIT off.")

# Release the previous model when rerunning this cell.
previous_engine = globals().pop("engine", None)
if previous_engine is not None:
    previous_engine.close()
    del previous_engine
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Device:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU (slower)")
print("PyTorch:", torch.__version__, "| Transformers:", version("transformers"))
engine = DecisionEngine.from_pretrained(
    MODEL_ID,
    backend="transformers",
    revision=MODEL_REVISION or None,
    device=DEVICE,
    vision=ENABLE_IMAGES,
    load_in_4bit=USE_4BIT,
    score_mode="full",
    n_ctx=8192,
)
print("Ready:", MODEL_ID)

## 3. Route a support message

`Choice` selects a team, `Noul` returns support for a yes/no question, and `Score` gives an expected rating over ordered labels. Change `MESSAGE` to try your own input.

In [ ]:
# @title Your message
MESSAGE = "Help! My payouts have been failing for three days."  # @param {type:"string"}

questions = {
    "department": Choice(
        "Which team should handle this?",
        {
            "billing": "Payments, invoicing, refunds",
            "technical": "Bugs, outages, integrations",
            "sales": "Pricing, upgrades, new accounts",
        },
    ),
    "urgent": Noul(
        "Does the message convey urgency?",
        {"true": "Explicitly time-sensitive", "false": "No urgency expressed"},
    ),
    "frustration": Score("How frustrated is the customer?", ["Calm", "Frustrated", "Very angry"]),
}
text_result = engine.decide(MESSAGE, questions)
answers = text_result.answers
print("Team:", answers["department"]["choice"])
print("Urgency support:", round(answers["urgent"]["noul"], 4))
print(
    "Frustration (0=calm, 1=frustrated, 2=very angry):", round(answers["frustration"]["score"], 3)
)
print("Team distribution:", json.dumps(answers["department"]["probabilities"], indent=2))
print("Reused input tokens:", text_result.usage.reused_input_tokens)
print("Generated tokens:", text_result.usage.generated_tokens)

### What these numbers mean

The default raw support is the full-vocabulary next-token probability `P("yes")`. A choice distribution normalizes the independently scored candidates; it is **not a calibrated probability of correctness**. Low support for every candidate can still produce a confident-looking normalized choice. Inspect the raw evidence below before choosing a threshold for a real workflow.

In [ ]:
print(json.dumps(text_result.answers["department"]["candidates"], indent=2))

## 4. Ask questions about an image

The default generates a red square so **Run all** needs no uploads. Enable `UPLOAD_YOUR_IMAGE` to open Colab's file picker, or enter an `IMAGE_PATH` from the runtime's Files panel. Uploaded images are processed inside your Colab runtime.

Image questions require `ENABLE_IMAGES=True` and a supported Qwen3.5 model. Large images count toward the context limit.

In [ ]:
# @title Image input
UPLOAD_YOUR_IMAGE = False  # @param {type:"boolean"}
IMAGE_PATH = ""  # @param {type:"string"}

image_result = None
if not ENABLE_IMAGES:
    print("Image example skipped. Enable images in the model settings to run it.")
else:
    if UPLOAD_YOUR_IMAGE:
        from google.colab import files

        uploaded = files.upload()
        if not uploaded:
            raise ValueError("No image selected. Disable uploads to use the sample.")
        image_path = Path(next(iter(uploaded)))
    elif IMAGE_PATH:
        image_path = Path(IMAGE_PATH)
    else:
        image_path = Path("openjev-red-square.png")
        Image.new("RGB", (112, 112), (255, 0, 0)).save(image_path)

    with Image.open(image_path) as source:
        preview = source.convert("RGB")
        preview.thumbnail((480, 480))
        display(preview)

    image_result = engine.decide(
        "Inspect the supplied image.",
        {
            "main_color": Choice(
                "Which color covers most of the image?",
                {"red": "Red", "blue": "Blue", "green": "Green", "other": "Another color"},
            ),
            "has_text": Noul("Does the image contain readable letters or words?"),
        },
        images=[str(image_path)],
    )
    print("Main color:", image_result.answers["main_color"]["choice"])
    print("Visible writing support:", round(image_result.answers["has_text"]["noul"], 4))
    print("Reused input tokens:", image_result.usage.reused_input_tokens)

## 5. Measure prefix caching

Compare the same text questions with caching on and off. Both paths are warmed before timing. This measures computation reuse, not accuracy, and excludes model loading. Speedup depends on the device and input; short prompts can be slower when copying caches costs more than recomputing them.

The benchmark also reports score differences. Different execution shapes and quantization can change floating-point results. On CPU it uses one measured iteration to keep this example shorter.

In [ ]:
cache_report = benchmark(
    engine,
    {"state": MESSAGE, "questions": questions},
    images=[],
    iterations=3 if DEVICE == "cuda" else 1,
)
print("Cached:  ", round(cache_report["cached_median_seconds"] * 1000, 1), "ms")
print("Uncached:", round(cache_report["uncached_median_seconds"] * 1000, 1), "ms")
print("Observed speedup:", round(cache_report["observed_speedup"], 2), "x")
print("Maximum candidate support difference:", cache_report["max_candidate_support_difference"])
print("Same choices:", cache_report["same_choices"])
print("Tokens reused:", cache_report["cached_usage"]["reused_input_tokens"])

## 6. Save results and release the model

The next cell saves the full scores and benchmark to `openjev-results.json`. Download it from Colab's Files panel if wanted. To run more examples after cleanup, rerun the model-loading cell.

In [ ]:
saved_results = {
    "text": text_result.to_dict(),
    "image": image_result.to_dict() if image_result is not None else None,
    "benchmark": cache_report,
}
output_path = Path("openjev-results.json")
output_path.write_text(json.dumps(saved_results, indent=2, allow_nan=False))
print("Saved:", output_path.resolve())
engine.close()
del engine
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

You can reuse the `questions` dictionaries in your own app. For Apple Silicon and GGUF setup, see the [README](https://github.com/mjdileep/OpenJev). Model weights retain their original license; OpenJev is MIT licensed.

Colab help: [runtime and GPU availability](https://research.google.com/colaboratory/faq.html).

Validated end-to-end on a Colab Tesla T4 on 2026-09-21 with both default and NF4 weights. See the [recorded results](https://github.com/mjdileep/OpenJev/blob/main/docs/validation.md#colab-cuda-notebook).